# Three-model agreement/disagreement analysis

This notebook reproduces the analysis and figures from the original three-model pipeline without loading or calling Qwen, Llama, or Mistral.

## Required input

One full candidate-level CSV, read directly from the original pipeline output:

`matched_ukb_full_final_since_2014_three_model_combined_labels.csv`

It must contain `id`, `title`, `abstract`, `year`, and the saved label/parse columns for all three models. The TRUE-agreement-only export is not sufficient because most analyses compare TRUE agreement with the remaining candidates.

## Outputs

Every derived table is saved as CSV and every figure as PNG and PDF. Figures are also displayed inline so they remain visible when the executed notebook is shared.

## 1. Check analysis dependencies

Run this notebook in the local analysis environment. Install missing packages there
before rerunning; this notebook does not install or upgrade packages automatically.
No tagging models or API credentials are required.


In [ ]:
from importlib.util import find_spec

packages = {
    "pandas": "pandas", "numpy": "numpy", "matplotlib": "matplotlib",
    "seaborn": "seaborn", "sklearn": "scikit-learn",
    "sentence_transformers": "sentence-transformers",
}
missing_packages = [package for module, package in packages.items() if find_spec(module) is None]
if missing_packages:
    raise ModuleNotFoundError(
        "Missing analysis dependencies. Install in this notebook's environment with: "
        "python -m pip install " + " ".join(missing_packages)
    )
print("Analysis dependencies available.")


## 2. Set local paths

Set `UKB_COMBINED_LABELS_CSV` to the full candidate-level combined labels CSV before
launching the notebook or shell runner. Both the original long filename above and
`three_model_combined_labels.csv` from the current tagging pipeline are supported.
The Showcase+ publication parquet and TRUE-agreement-only export are not substitutes.
Tables and figures are written under the repository's `output/` directory.


In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "utils").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from utils import shared_paths as P
from utils.shared_analysis_window import filter_analysis_window
P.bootstrap()

input_value = os.environ.get("UKB_COMBINED_LABELS_CSV", "").strip()
if not input_value:
    input_names = {
        "matched_ukb_full_final_since_2014_three_model_combined_labels.csv",
        "three_model_combined_labels.csv",
    }
    candidates = sorted(path for path in P.DATA.rglob("*combined_labels.csv") if path.name in input_names)
    if len(candidates) == 1:
        input_value = str(candidates[0])
    elif len(candidates) > 1:
        raise ValueError("Multiple combined-label CSVs found; set UKB_COMBINED_LABELS_CSV explicitly.")
    else:
        raise FileNotFoundError(
            "Missing full three-model combined-labels CSV; set UKB_COMBINED_LABELS_CSV. "
            "The Showcase+ parquet cannot replace this input."
        )
INPUT_PATH = Path(input_value).expanduser().resolve()
OUTPUT_DIR = P.TABLE_DATA_ANALYSIS / "00_dataset" / "three_model_agreement"
FIGURE_DIR = P.FIG_DATA_ANALYSIS / "00_dataset" / "three_model_agreement"

RUN_SEMANTIC_ANALYSIS = True
MAX_TFIDF_PER_GROUP = 20_000
MAX_SEMANTIC_PER_GROUP = 3_000
SEED = 42

LCDS_PALETTE = ["#344874", "#FFBB00", "#5B8DB8", "#2A9D8F", "#E76F51", "#8E6C9E"]

if not INPUT_PATH.is_file():
    raise FileNotFoundError(
        f"Combined consensus CSV not found: {INPUT_PATH}\n"
        "Check UKB_COMBINED_LABELS_CSV and the original three-model pipeline output."
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input: {INPUT_PATH}")
print(f"Input size: {INPUT_PATH.stat().st_size / 1024**2:.1f} MB")
print(f"Outputs: {OUTPUT_DIR}")
print(f"Figures: {FIGURE_DIR}")


## 3. Imports and reusable helpers

In [ ]:
import gc
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from utils.shared_style import PNG_DPI
from utils.shared_style import apply_typography, finalize_figure, set_title
apply_typography()
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")
apply_typography()

MODEL_NAMES = ["qwen", "llama3_8b", "mistral_7b"]
MODEL_DISPLAY = {"qwen": "Qwen2.5-7B", "llama3_8b": "Llama3-8B", "mistral_7b": "Mistral-7B"}


def normalise_label(series):
    result = pd.Series(pd.NA, index=series.index, dtype="Int64")
    numeric = pd.to_numeric(series, errors="coerce")
    result.loc[numeric.eq(1)] = 1
    result.loc[numeric.eq(0)] = 0
    text = series.astype(str).str.strip().str.lower()
    result.loc[text.isin(["true", "yes", "y", "1", "1.0"])] = 1
    result.loc[text.isin(["false", "no", "n", "0", "0.0"])] = 0
    return result


def normalise_bool(series):
    if series.dtype == bool:
        return series.fillna(False).astype(bool)
    text = series.astype(str).str.strip().str.lower()
    return text.map({
        "true": True, "1": True, "1.0": True, "yes": True, "y": True,
        "false": False, "0": False, "0.0": False, "no": False, "n": False,
        "nan": False, "none": False, "<na>": False, "": False,
    }).fillna(False).astype(bool)


def clean_year(series):
    years = pd.to_numeric(series, errors="coerce")
    return years.where(years.between(1900, 2035)).astype("Int64")


def analysis_text(frame):
    return (
        frame["title"].fillna("").astype(str).str.strip()
        + "\n"
        + frame["abstract"].fillna("").astype(str).str.strip()
    )


def save_table(frame, filename):
    path = OUTPUT_DIR / filename
    frame.to_csv(path, index=False)
    print(f"Saved table: {path}")
    return path


def save_figure(figure, filename):
    finalize_figure(figure)
    figure.tight_layout()
    png_path = FIGURE_DIR / f"{filename}.png"
    pdf_path = FIGURE_DIR / f"{filename}.pdf"
    figure.savefig(png_path, dpi=PNG_DPI, bbox_inches="tight")
    figure.savefig(pdf_path, bbox_inches="tight")
    display(figure)
    plt.close(figure)
    print(f"Saved figure: {png_path}")
    return png_path

## 4. Load, validate, and rebuild consensus fields

**Outputs:**

- `combined_consensus_normalised.csv`
- `three_model_TRUE_agreement.csv`
- `rest_NOT_three_model_TRUE_agreement.csv`

In [ ]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Combined consensus CSV not found: {INPUT_PATH}\n"
        "Update INPUT_PATH in Section 2. Do not use the TRUE-agreement-only export."
    )

combined = pd.read_csv(INPUT_PATH, low_memory=False)
required = {"id", "title", "abstract", "year"}
for model in MODEL_NAMES:
    required.update({f"{model}_label", f"{model}_parse_ok"})

missing = sorted(required - set(combined.columns))
if missing:
    raise ValueError(f"The combined consensus CSV is missing required columns: {missing}")

combined["id"] = combined["id"].astype(str).str.strip()
combined = combined[combined["id"].ne("")].drop_duplicates("id", keep="first").reset_index(drop=True)
combined["title"] = combined["title"].fillna("").astype(str)
combined["abstract"] = combined["abstract"].fillna("").astype(str)
combined["year_int"] = clean_year(combined["year"])
combined = filter_analysis_window(combined, year_col="year_int")

for model in MODEL_NAMES:
    combined[f"{model}_label"] = normalise_label(combined[f"{model}_label"])
    combined[f"{model}_parse_ok"] = normalise_bool(combined[f"{model}_parse_ok"])
    combined[f"{model}_true"] = combined[f"{model}_parse_ok"] & combined[f"{model}_label"].eq(1)
    combined[f"{model}_false"] = combined[f"{model}_parse_ok"] & combined[f"{model}_label"].eq(0)

combined["n_models_parsed"] = sum(combined[f"{model}_parse_ok"].astype(int) for model in MODEL_NAMES)
combined["n_true_votes"] = sum(combined[f"{model}_true"].astype(int) for model in MODEL_NAMES)
combined["n_false_votes"] = sum(combined[f"{model}_false"].astype(int) for model in MODEL_NAMES)
combined["all_three_parsed"] = combined["n_models_parsed"].eq(3)
combined["three_model_TRUE_agreement"] = np.logical_and.reduce(
    [combined[f"{model}_true"] for model in MODEL_NAMES]
)
combined["three_model_FALSE_agreement"] = np.logical_and.reduce(
    [combined[f"{model}_false"] for model in MODEL_NAMES]
)


def vote_signature(row):
    values = []
    for model in MODEL_NAMES:
        if not row[f"{model}_parse_ok"]:
            value = "NA"
        else:
            value = "T" if row[f"{model}_label"] == 1 else "F"
        values.append(f"{model}={value}")
    return " | ".join(values)


def consensus_group(row):
    if row["three_model_TRUE_agreement"]:
        return "Three-model TRUE agreement"
    if row["three_model_FALSE_agreement"]:
        return "Three-model FALSE agreement"
    if row["n_true_votes"] == 2:
        return "Two TRUE votes"
    if row["n_true_votes"] == 1:
        return "One TRUE vote"
    if row["n_models_parsed"] > 0:
        return "No TRUE votes / parsed non-positive"
    return "No parsed model labels"


combined["vote_signature"] = combined.apply(vote_signature, axis=1)
combined["consensus_group"] = combined.apply(consensus_group, axis=1)

three_true = combined[combined["three_model_TRUE_agreement"]].copy()
rest = combined[~combined["three_model_TRUE_agreement"]].copy()

if three_true.empty or rest.empty:
    raise ValueError(
        "Both consensus groups are required. This appears to be a subset export. "
        "Use matched_ukb_full_final_since_2014_three_model_combined_labels.csv."
    )

save_table(combined, "combined_consensus_normalised.csv")
save_table(three_true, "three_model_TRUE_agreement.csv")
save_table(rest, "rest_NOT_three_model_TRUE_agreement.csv")

print(f"Candidates: {len(combined):,}")
print(f"Three-model TRUE agreement: {len(three_true):,}")
print(f"Rest: {len(rest):,}")
display(combined[["id", "year_int", "n_true_votes", "consensus_group"]].head())

## 5. Dataset overview

**Output:** `dataset_overview.csv`

In [ ]:
overview = pd.DataFrame([
    {"metric": "total_candidates", "value": len(combined)},
    {"metric": "unique_ids", "value": combined["id"].nunique()},
    {"metric": "three_model_TRUE_agreement", "value": len(three_true)},
    {"metric": "rest_NOT_three_model_TRUE_agreement", "value": len(rest)},
    {"metric": "three_model_TRUE_agreement_percent", "value": 100 * len(three_true) / len(combined)},
    {"metric": "three_model_FALSE_agreement", "value": int(combined["three_model_FALSE_agreement"].sum())},
    {"metric": "all_three_parsed", "value": int(combined["all_three_parsed"].sum())},
    {"metric": "earliest_year", "value": combined["year_int"].min()},
    {"metric": "latest_year", "value": combined["year_int"].max()},
])
save_table(overview, "dataset_overview.csv")
display(overview)

## 6. Consensus statistics and distributions

**Outputs:**

- `model_positive_rate_summary.csv`
- `true_vote_distribution.csv`
- `consensus_group_distribution.csv`
- `vote_signature_distribution.csv`

In [ ]:
model_summary = []
for model in MODEL_NAMES:
    parsed = int(combined[f"{model}_parse_ok"].sum())
    true = int(combined[f"{model}_true"].sum())
    false = int(combined[f"{model}_false"].sum())
    model_summary.append({
        "model": model,
        "display_name": MODEL_DISPLAY[model],
        "parsed": parsed,
        "true": true,
        "false": false,
        "parse_rate_percent": 100 * parsed / len(combined),
        "true_percent_among_parsed": 100 * true / max(parsed, 1),
    })

model_summary = pd.DataFrame(model_summary)
vote_distribution = combined["n_true_votes"].value_counts().sort_index().rename_axis("n_true_votes").reset_index(name="n_candidates")
group_distribution = combined["consensus_group"].value_counts().rename_axis("consensus_group").reset_index(name="n_candidates")
signature_distribution = combined["vote_signature"].value_counts().rename_axis("vote_signature").reset_index(name="n_candidates")

save_table(model_summary, "model_positive_rate_summary.csv")
save_table(vote_distribution, "true_vote_distribution.csv")
save_table(group_distribution, "consensus_group_distribution.csv")
save_table(signature_distribution, "vote_signature_distribution.csv")

display(model_summary)
display(vote_distribution)
display(group_distribution)
display(signature_distribution)

## 7. Model positive rates

**Figure:** `model_positive_rate_summary`

In [ ]:
figure, axis = plt.subplots(figsize=(8.5, 5))
bars = axis.bar(model_summary["display_name"], model_summary["true_percent_among_parsed"], color=LCDS_PALETTE[:3])
set_title(axis, 'Model-level positive rate under the P2 prompt')
axis.set(ylabel='TRUE among parsed rows (%)')
axis.grid(axis="y", alpha=0.25)
axis.bar_label(bars, fmt="%.1f%%", padding=3)
save_figure(figure, "model_positive_rate_summary")

## 8. TRUE-vote distribution

**Figure:** `true_vote_distribution`

In [ ]:
figure, axis = plt.subplots(figsize=(8, 5))
bars = axis.bar(vote_distribution["n_true_votes"].astype(str), vote_distribution["n_candidates"], color=LCDS_PALETTE[0])
set_title(axis, 'TRUE-vote distribution across Qwen, Llama3-8B, and Mistral-7B')
axis.set(xlabel='Number of models predicting TRUE', ylabel='Number of candidates')
axis.grid(axis="y", alpha=0.25)
axis.bar_label(bars, labels=[f"{value:,}" for value in vote_distribution["n_candidates"]], padding=3)
save_figure(figure, "true_vote_distribution")

## 9. Consensus-group distribution

**Figure:** `consensus_group_distribution`

In [ ]:
plot_data = group_distribution.sort_values("n_candidates")
figure, axis = plt.subplots(figsize=(10, 6))
bars = axis.barh(plot_data["consensus_group"], plot_data["n_candidates"], color=LCDS_PALETTE[2])
set_title(axis, 'Consensus-group distribution')
axis.set(xlabel='Number of candidates')
axis.grid(axis="x", alpha=0.25)
axis.bar_label(bars, labels=[f"{value:,}" for value in plot_data["n_candidates"]], padding=3)
save_figure(figure, "consensus_group_distribution")

## 10. Pairwise model agreement

**Outputs:** `pairwise_model_agreement.csv` and `pairwise_model_agreement_heatmap`

In [ ]:
agreement_matrix = pd.DataFrame(np.eye(len(MODEL_NAMES)), index=MODEL_NAMES, columns=MODEL_NAMES)
pairwise_rows = []

for index, model_a in enumerate(MODEL_NAMES):
    for model_b in MODEL_NAMES[index + 1:]:
        mask = combined[f"{model_a}_parse_ok"] & combined[f"{model_b}_parse_ok"]
        agreement = (
            combined.loc[mask, f"{model_a}_label"].astype(int).to_numpy()
            == combined.loc[mask, f"{model_b}_label"].astype(int).to_numpy()
        ).mean()
        agreement_matrix.loc[model_a, model_b] = agreement
        agreement_matrix.loc[model_b, model_a] = agreement
        pairwise_rows.append({
            "model_a": model_a,
            "model_b": model_b,
            "n_both_parsed": int(mask.sum()),
            "agreement_percent": 100 * agreement,
        })

pairwise_agreement = pd.DataFrame(pairwise_rows)
save_table(pairwise_agreement, "pairwise_model_agreement.csv")
display(pairwise_agreement)

figure, axis = plt.subplots(figsize=(7, 6))
sns.heatmap(
    100 * agreement_matrix,
    annot=True,
    fmt=".1f",
    cmap=sns.light_palette(LCDS_PALETTE[0], as_cmap=True),
    vmin=0,
    vmax=100,
    square=True,
    cbar_kws={"label": "Agreement (%)"},
    ax=axis,
)
set_title(axis, "Pairwise agreement among parsed model labels")
save_figure(figure, "pairwise_model_agreement_heatmap")

## 11. Yearly consensus table and TRUE-versus-rest counts

**Outputs:** `yearly_three_model_consensus_trend.csv` and `yearly_TRUE_vs_rest_logscale`

In [ ]:
year_data = combined.dropna(subset=["year_int"]).copy()
year_data["year_int"] = year_data["year_int"].astype(int)

yearly = year_data.groupby("year_int").agg(
    total_candidates=("id", "count"),
    three_model_TRUE_agreement=("three_model_TRUE_agreement", "sum"),
    three_model_FALSE_agreement=("three_model_FALSE_agreement", "sum"),
    qwen_TRUE=("qwen_true", "sum"),
    llama3_8b_TRUE=("llama3_8b_true", "sum"),
    mistral_7b_TRUE=("mistral_7b_true", "sum"),
).reset_index()
yearly["rest_NOT_three_model_TRUE_agreement"] = yearly["total_candidates"] - yearly["three_model_TRUE_agreement"]
yearly["three_model_TRUE_agreement_rate_percent"] = 100 * yearly["three_model_TRUE_agreement"] / yearly["total_candidates"]
yearly["rest_rate_percent"] = 100 - yearly["three_model_TRUE_agreement_rate_percent"]

save_table(yearly, "yearly_three_model_consensus_trend.csv")
display(yearly)

figure, axis = plt.subplots(figsize=(11, 5.8))
axis.plot(yearly["year_int"], yearly["three_model_TRUE_agreement"], marker="o", color=LCDS_PALETTE[0], label="Three-model TRUE agreement")
axis.plot(yearly["year_int"], yearly["rest_NOT_three_model_TRUE_agreement"], marker="o", color=LCDS_PALETTE[4], label="Rest")
axis.set_yscale("log")
set_title(axis, 'Yearly consensus counts')
axis.set(xlabel='Publication year', ylabel='Candidates (log scale)')
axis.legend()
save_figure(figure, "yearly_TRUE_vs_rest_logscale")

## 12. Yearly three-model TRUE-agreement rate

**Figure:** `yearly_three_model_TRUE_agreement_rate`

In [ ]:
figure, axis = plt.subplots(figsize=(11, 5.8))
axis.plot(yearly["year_int"], yearly["three_model_TRUE_agreement_rate_percent"], marker="o", color=LCDS_PALETTE[3])
set_title(axis, 'Yearly three-model TRUE-agreement rate')
axis.set(xlabel='Publication year', ylabel='TRUE agreement among candidates (%)')
save_figure(figure, "yearly_three_model_TRUE_agreement_rate")

## 13. Yearly positive predictions by model

**Figure:** `yearly_model_TRUE_counts`

In [ ]:
figure, axis = plt.subplots(figsize=(11, 5.8))
for model, color in zip(MODEL_NAMES, LCDS_PALETTE[:3]):
    axis.plot(yearly["year_int"], yearly[f"{model}_TRUE"], marker="o", color=color, label=MODEL_DISPLAY[model])
set_title(axis, 'Yearly positive predictions by model')
axis.set(xlabel='Publication year', ylabel='TRUE predictions')
axis.legend()
save_figure(figure, "yearly_model_TRUE_counts")

## 14. Explicit UK Biobank mentions

**Outputs:** `explicit_ukb_mention_summary.csv` and `explicit_ukb_mention_by_binary_split`

In [ ]:
ukb_pattern = re.compile(
    r"\b(uk\s*biobank|u\.?k\.?\s*biobank|united\s+kingdom\s+biobank|ukb|ukbb)\b",
    flags=re.I,
)
combined["analysis_text"] = analysis_text(combined)
combined["explicit_ukb_title_or_abstract"] = combined["analysis_text"].str.contains(ukb_pattern, na=False)
combined["binary_split"] = np.where(
    combined["three_model_TRUE_agreement"],
    "Three-model TRUE agreement",
    "Rest: not three-model TRUE agreement",
)

explicit_summary = combined.groupby("binary_split").agg(
    n=("id", "count"),
    explicit_ukb=("explicit_ukb_title_or_abstract", "sum"),
).reset_index()
explicit_summary["explicit_ukb_percent"] = 100 * explicit_summary["explicit_ukb"] / explicit_summary["n"]
save_table(explicit_summary, "explicit_ukb_mention_summary.csv")
display(explicit_summary)

figure, axis = plt.subplots(figsize=(9, 5))
bars = axis.bar(explicit_summary["binary_split"], explicit_summary["explicit_ukb_percent"], color=[LCDS_PALETTE[0], LCDS_PALETTE[4]])
set_title(axis, 'Explicit UK Biobank mention by consensus split')
axis.set(ylabel='Explicit mention in title or abstract (%)')
axis.tick_params(axis="x", rotation=12)
axis.bar_label(bars, fmt="%.1f%%", padding=3)
save_figure(figure, "explicit_ukb_mention_by_binary_split")

## 15. Keyword-category profile

**Outputs:** `keyword_category_summary.csv` and `keyword_category_profile_by_binary_split`

In [ ]:
CATEGORY_PATTERNS = {
    "explicit UKB": r"\b(uk\s*biobank|u\.?k\.?\s*biobank|united\s+kingdom\s+biobank|ukb|ukbb)\b",
    "generic biobank": r"\b(biobank|biobanking|biobanks)\b",
    "UK population cue": r"\b(uk|british|england|scotland|wales|united kingdom)\b",
    "cohort / participants": r"\b(cohort|participants|population[- ]based|prospective|baseline assessment)\b",
    "genetics / genomics": r"\b(genetic|genomic|genotype|genotyping|gwas|polygenic|exome|sequencing)\b",
    "imaging": r"\b(imaging|mri|brain imaging|cardiac imaging|radiomics)\b",
    "linked records / EHR": r"\b(linked records|hospital episode statistics|hes|electronic health records|ehr|registry|registries)\b",
    "machine learning": r"\b(machine learning|deep learning|artificial intelligence|neural network|prediction model)\b",
    "cardiometabolic": r"\b(cardiovascular|heart|diabetes|obesity|metabolic|hypertension)\b",
    "cancer": r"\b(cancer|tumou?r|oncology|carcinoma|neoplasm)\b",
    "mental health / brain": r"\b(depression|anxiety|psychiatric|mental health|brain|cognition|dementia)\b",
    "other named biobank": r"\b(china kadoorie|biobank japan|finn?gen|all of us|million veteran|lifelines|decode|cartagene)\b",
}

category_rows = []
for group_name, group in combined.groupby("binary_split"):
    text = analysis_text(group)
    for category, pattern in CATEGORY_PATTERNS.items():
        hits = text.str.contains(pattern, case=False, regex=True, na=False)
        category_rows.append({
            "binary_split": group_name,
            "category": category,
            "n_group": len(group),
            "n_with_category": int(hits.sum()),
            "percent_with_category": 100 * hits.mean(),
        })

category_summary = pd.DataFrame(category_rows)
save_table(category_summary, "keyword_category_summary.csv")
display(category_summary)

pivot = category_summary.pivot(index="category", columns="binary_split", values="percent_with_category").fillna(0)
true_column = "Three-model TRUE agreement"
rest_column = "Rest: not three-model TRUE agreement"
pivot = pivot.sort_values(true_column)
y = np.arange(len(pivot))

figure, axis = plt.subplots(figsize=(10, 8))
axis.barh(y - 0.19, pivot[true_column], height=0.38, color=LCDS_PALETTE[0], label=true_column)
axis.barh(y + 0.19, pivot[rest_column], height=0.38, color=LCDS_PALETTE[4], label=rest_column)
axis.set_yticks(y, pivot.index)
set_title(axis, 'Keyword-category profile by consensus split')
axis.set(xlabel='Papers with cue (%)')
axis.legend()
save_figure(figure, "keyword_category_profile_by_binary_split")

## 16. TF-IDF discriminative terms

**Outputs:**

- `tfidf_discriminative_terms.csv`
- `tfidf_terms_associated_with_three_model_TRUE_agreement`
- `tfidf_terms_associated_with_rest`

In [ ]:
tfidf_true = three_true.sample(n=min(len(three_true), MAX_TFIDF_PER_GROUP), random_state=SEED)
tfidf_rest = rest.sample(n=min(len(rest), MAX_TFIDF_PER_GROUP), random_state=SEED)
tfidf_data = pd.concat([
    tfidf_true.assign(binary_split="Three-model TRUE agreement"),
    tfidf_rest.assign(binary_split="Rest: not three-model TRUE agreement"),
], ignore_index=True)
tfidf_data["analysis_text"] = analysis_text(tfidf_data)

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=5 if len(tfidf_data) >= 1_000 else 2,
    max_df=0.85,
    max_features=12_000,
)
matrix = vectorizer.fit_transform(tfidf_data["analysis_text"])
terms = np.asarray(vectorizer.get_feature_names_out())
true_mask = tfidf_data["binary_split"].eq("Three-model TRUE agreement").to_numpy()
true_mean = np.asarray(matrix[true_mask].mean(axis=0)).ravel()
rest_mean = np.asarray(matrix[~true_mask].mean(axis=0)).ravel()

tfidf_terms = pd.DataFrame({
    "term": terms,
    "mean_tfidf_three_model_TRUE_agreement": true_mean,
    "mean_tfidf_rest": rest_mean,
    "difference_TRUE_minus_rest": true_mean - rest_mean,
}).sort_values("difference_TRUE_minus_rest", ascending=False)
save_table(tfidf_terms, "tfidf_discriminative_terms.csv")

top_true = tfidf_terms.head(25).iloc[::-1]
top_rest = tfidf_terms.tail(25).sort_values("difference_TRUE_minus_rest", ascending=True).iloc[::-1]
display(tfidf_terms.head(30))
display(tfidf_terms.tail(30).sort_values("difference_TRUE_minus_rest"))

figure, axis = plt.subplots(figsize=(10, 8))
axis.barh(top_true["term"], top_true["difference_TRUE_minus_rest"], color=LCDS_PALETTE[3])
set_title(axis, 'Terms associated with three-model TRUE agreement')
axis.set(xlabel='Mean TF-IDF difference: TRUE minus rest')
save_figure(figure, "tfidf_terms_associated_with_three_model_TRUE_agreement")

figure, axis = plt.subplots(figsize=(10, 8))
axis.barh(top_rest["term"], -top_rest["difference_TRUE_minus_rest"], color=LCDS_PALETTE[4])
set_title(axis, 'Terms associated with the remaining candidates')
axis.set(xlabel='Mean TF-IDF difference: rest minus TRUE')
save_figure(figure, "tfidf_terms_associated_with_rest")

## 17. Semantic separation diagnostic

This analysis uses the public `all-MiniLM-L6-v2` sentence-embedding model. It does not call any tagging model. Set `RUN_SEMANTIC_ANALYSIS=False` in Section 2 to skip it.

**Outputs:**

- `semantic_sample_with_coordinates.csv`
- `semantic_metrics.csv`
- `semantic_map_TRUE_agreement_vs_rest`
- `semantic_map_by_year`

In [ ]:
if RUN_SEMANTIC_ANALYSIS:
    from sentence_transformers import SentenceTransformer

    semantic_true = three_true.sample(n=min(len(three_true), MAX_SEMANTIC_PER_GROUP), random_state=SEED)
    semantic_rest = rest.sample(n=min(len(rest), MAX_SEMANTIC_PER_GROUP), random_state=SEED)
    semantic_data = pd.concat([
        semantic_true.assign(binary_split="Three-model TRUE agreement"),
        semantic_rest.assign(binary_split="Rest: not three-model TRUE agreement"),
    ], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
    semantic_data["analysis_text"] = analysis_text(semantic_data).str.slice(0, 3_500)
    labels = semantic_data["binary_split"].eq("Three-model TRUE agreement").astype(int).to_numpy()

    embedding_method = "sentence-transformers/all-MiniLM-L6-v2"
    try:
        embedder = SentenceTransformer(embedding_method)
        embeddings = embedder.encode(
            semantic_data["analysis_text"].tolist(),
            batch_size=128,
            show_progress_bar=True,
            normalize_embeddings=True,
        )
        del embedder
        gc.collect()
    except Exception as error:
        print(f"Sentence embeddings unavailable ({error}); using TF-IDF/SVD fallback.")
        fallback = TfidfVectorizer(
            lowercase=True,
            stop_words="english",
            ngram_range=(1, 2),
            min_df=3,
            max_df=0.9,
            max_features=12_000,
        )
        text_matrix = fallback.fit_transform(semantic_data["analysis_text"])
        components = min(100, text_matrix.shape[0] - 1, text_matrix.shape[1] - 1)
        embeddings = TruncatedSVD(n_components=components, random_state=SEED).fit_transform(text_matrix)
        embeddings /= np.maximum(np.linalg.norm(embeddings, axis=1, keepdims=True), 1e-12)
        embedding_method = "TF-IDF + TruncatedSVD fallback"

    silhouette = silhouette_score(embeddings, labels, metric="cosine")
    true_centroid = embeddings[labels == 1].mean(axis=0)
    rest_centroid = embeddings[labels == 0].mean(axis=0)
    true_centroid /= max(np.linalg.norm(true_centroid), 1e-12)
    rest_centroid /= max(np.linalg.norm(rest_centroid), 1e-12)
    centroid_similarity = float(np.dot(true_centroid, rest_centroid))

    coordinates = PCA(n_components=2, random_state=SEED).fit_transform(embeddings)
    semantic_data[["semantic_x", "semantic_y"]] = coordinates
    semantic_metrics = pd.DataFrame([
        {"metric": "embedding_method", "value": embedding_method},
        {"metric": "semantic_sample_size", "value": len(semantic_data)},
        {"metric": "semantic_sample_three_model_TRUE_agreement", "value": int(labels.sum())},
        {"metric": "semantic_sample_rest", "value": int((labels == 0).sum())},
        {"metric": "SI_silhouette_index_cosine", "value": silhouette},
        {"metric": "centroid_cosine_similarity", "value": centroid_similarity},
        {"metric": "centroid_cosine_distance", "value": 1 - centroid_similarity},
    ])
    save_table(semantic_data, "semantic_sample_with_coordinates.csv")
    save_table(semantic_metrics, "semantic_metrics.csv")
    display(semantic_metrics)

    figure, axis = plt.subplots(figsize=(9, 7))
    for group, color in zip(
        ["Rest: not three-model TRUE agreement", "Three-model TRUE agreement"],
        [LCDS_PALETTE[4], LCDS_PALETTE[0]],
    ):
        subset = semantic_data[semantic_data["binary_split"].eq(group)]
        axis.scatter(subset["semantic_x"], subset["semantic_y"], s=12, alpha=0.5, color=color, label=f"{group} (n={len(subset):,})")
    set_title(axis, f'Semantic map: TRUE agreement versus rest\nSilhouette index = {silhouette:.4f}')
    axis.set(xlabel='Semantic component 1', ylabel='Semantic component 2')
    axis.legend()
    save_figure(figure, "semantic_map_TRUE_agreement_vs_rest")

    year_mask = semantic_data["year_int"].notna()
    figure, axis = plt.subplots(figsize=(9, 7))
    points = axis.scatter(
        semantic_data.loc[year_mask, "semantic_x"],
        semantic_data.loc[year_mask, "semantic_y"],
        c=semantic_data.loc[year_mask, "year_int"].astype(int),
        cmap="viridis",
        s=12,
        alpha=0.6,
    )
    set_title(axis, 'Semantic map coloured by publication year')
    axis.set(xlabel='Semantic component 1', ylabel='Semantic component 2')
    figure.colorbar(points, ax=axis, label="Publication year")
    save_figure(figure, "semantic_map_by_year")
else:
    print("Semantic analysis skipped by configuration.")

## 18. Output inventory

This final cell confirms which tables and figures were created.

In [ ]:
table_files = sorted(OUTPUT_DIR.glob("*.csv"))
figure_files = sorted(FIGURE_DIR.glob("*.png"))

inventory = pd.DataFrame(
    [{"type": "table", "file": path.name, "size_kb": path.stat().st_size / 1024} for path in table_files]
    + [{"type": "figure", "file": path.name, "size_kb": path.stat().st_size / 1024} for path in figure_files]
)
display(inventory)
print(f"Created {len(table_files)} CSV tables and {len(figure_files)} PNG figures.")
print(f"Output folder: {OUTPUT_DIR}")